In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/1429_1.csv"
df = pd.read_csv(path)

df.head()
df.columns
df.info()

In [ ]:
df = df[['reviews.text','reviews.rating','reviews.date','asins']]

df = df.rename(columns={
    'reviews.text':'review',
    'reviews.rating':'rating',
    'reviews.date':'date',
    'asins':'product_id'
})

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df['month'] = df['date'].dt.to_period('M')

In [ ]:
df[['review','month']].head()

In [ ]:
df = df.dropna(subset=['review'])

In [ ]:
import re

def clean_text(text):

    text = text.lower()
    text = re.sub(r"http\S+","",text)
    text = re.sub(r"[^a-zA-Z\s]","",text)

    return text

df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
df[['review','clean_review']].head()

**Phase** **2**

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [ ]:
def extract_aspects(text):

    doc = nlp(text)

    aspects = []

    for chunk in doc.noun_chunks:

        aspect = chunk.text.lower().strip()

        if len(aspect) > 3:
            aspects.append(aspect)

    return aspects

In [ ]:
sample_df = df.sample(5000, random_state=42)

sample_df['aspects'] = sample_df['clean_review'].apply(extract_aspects)

In [ ]:
sample_df = sample_df[sample_df['aspects'].map(len) > 0]

In [ ]:
sample_df = sample_df.explode('aspects')

In [ ]:
# normalize aspects
sample_df['aspects'] = sample_df['aspects'].str.lower().str.strip()

# remove articles
sample_df['aspects'] = sample_df['aspects'].str.replace(r'^(the|this|a|an)\s+', '', regex=True)

# remove very short terms
sample_df = sample_df[sample_df['aspects'].str.len() > 3]

# remove pronouns / meaningless terms
remove_terms = [
    "this","that","they","them","what","which",
    "something","anything","everything","someone",
    "one","some","lot","thing",
    "everything","nothing","anything"
]

sample_df = sample_df[~sample_df['aspects'].isin(remove_terms)]

In [ ]:
remove_context_words = [
    "kids","son","wife","gift","people","family",
    "my son","my kids","my wife"
]

sample_df = sample_df[~sample_df['aspects'].isin(remove_context_words)]

In [ ]:
remove_generic = [
    "product","device","thing","stuff","everything"
]

sample_df = sample_df[~sample_df['aspects'].isin(remove_generic)]

In [ ]:
remove_noise = [
    "these",
    "christmas",
    "money",
    "time",
    "anyone",
    "someone",
    "something",
    "everything",
    "great product",
    "great tablet",
    "my daughter",
    "my son",
    "my kids",
    "my wife"
]

sample_df = sample_df[~sample_df['aspects'].isin(remove_noise)]

In [ ]:
sample_df = sample_df[~sample_df['aspects'].str.startswith("my ")]

In [ ]:
# remove long noun phrases
sample_df = sample_df[sample_df['aspects'].str.split().str.len() <= 2]

In [ ]:
sample_df = sample_df[sample_df['aspects'].str.split().str.len() <= 3]

aspect_counts = sample_df['aspects'].value_counts()

valid_aspects = aspect_counts[aspect_counts > 30].index

sample_df = sample_df[sample_df['aspects'].isin(valid_aspects)]

In [ ]:
sample_df['aspects'].value_counts().head(20)

**PHASE 3**

In [ ]:
!pip install transformers torch tqdm

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/1429_1.csv")
df.head()

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/1429_1.csv"
df = pd.read_csv(path)

df.head()
df.columns
df.info()

In [ ]:
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
print(df.columns)

In [ ]:
df["reviewText"] = df["reviews.text"]

In [ ]:
df["reviewText"] = df["reviewText"].fillna("")
df["aspect"] = df["reviewText"].apply(extract_aspects)

In [ ]:
df['aspect'] = df['aspect'].apply(lambda x: x if isinstance(x, list) else [])
print(len(df))
# df = df[df['aspect'].map(len) > 0]


In [ ]:
df = df[df['aspect'].map(len) > 0]

In [ ]:
df["aspect"].head(10)

In [ ]:
df = df.explode("aspect")

In [ ]:
df["aspect_text"] = df["aspect"] + " : " + df["reviewText"]

In [ ]:
def predict_sentiment(texts, batch_size=32):

    all_scores = []

    for i in tqdm(range(0, len(texts), batch_size)):

        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128, # yahan raat ko change kiya tha
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(probs, dim=1)

        scores = preds.cpu().numpy() + 1   # convert 0-4 → 1-5 stars
        all_scores.extend(scores)

    return all_scores

In [ ]:
# texts = df["aspect_text"].tolist()

In [ ]:
# df["sentiment_score"] = predict_sentiment(texts)

In [ ]:
texts = df["aspect_text"].tolist()

df["sentiment_score"] = predict_sentiment(texts, batch_size=128) # yahan bhi 256 se 128

In [ ]:
df["aspect"] = df["aspect"].str.lower().str.strip()


In [ ]:
bad_aspects = [
    "this", "that", "they", "it", "we", "you",
    "thing", "time", "lot", "way", "people",
    "someone", "something", "product", "stuff"
]

df = df[~df["aspect"].isin(bad_aspects)]


In [ ]:
df = df[df["aspect"].str.len() > 2]


In [ ]:
top_aspects = df["aspect"].value_counts().head(20).index

df = df[df["aspect"].isin(top_aspects)]

In [ ]:
df["aspect"].value_counts().head(20)

In [ ]:
df["aspect"] = df["aspect"].apply(lambda x: x.split()[-1])

In [ ]:
bad_words = [
    "thing","something","everything","lot","which","what",
    "them","they","someone","son","kids","gift"
]

df = df[~df["aspect"].isin(bad_words)]


In [ ]:
df["aspect"].value_counts().head(20)

In [ ]:
remove_words = ["product", "amazon"]

df = df[~df["aspect"].isin(remove_words)]

In [ ]:
df["aspect"].value_counts().head(20)

In [ ]:
df[["aspect","sentiment_score"]].head(10)

In [ ]:
df["sentiment_score"].value_counts()

In [ ]:
df["reviews.date"] = pd.to_datetime(df["reviews.date"], errors="coerce")
df["reviews.date"] = df["reviews.date"].dt.strftime("%d-%m-%Y")
df["month"] = pd.to_datetime(df["reviews.date"], errors="coerce").dt.to_period("M")

In [ ]:
df[["reviews.date","month"]].head(10)

In [ ]:
aspect_month_sentiment = (
    df.groupby(["month", "aspect"])["sentiment_score"]
    .mean()
    .reset_index()
)

In [ ]:
aspect_month_sentiment.head(10)

In [ ]:
pivot_table = aspect_month_sentiment.pivot(
    index="month",
    columns="aspect",
    values="sentiment_score"
)


In [ ]:
pivot_table.head()

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=pivot_table)

In [ ]:
import pandas as pd

path = ""
df = pd.read_csv(path)

df = df[['name','reviews.text','reviews.rating','reviews.date']]
df.columns = ['Product_Name','review','rating','time/date']

In [ ]:
alexa_df = df[df['Product_Name'].str.contains("Alexa", case=False, na=False)].copy()

echo_df = df[df['Product_Name'].str.contains("Echo", case=False, na=False)].copy()

tablet_df = df[df['Product_Name'].str.contains("Tablet", case=False, na=False)].copy()

In [ ]:
def sentiment(r):
    if r >= 4:
        return 1
    elif r == 3:
        return 0
    else:
        return -1

for dataset in [alexa_df, echo_df, tablet_df]:
    dataset.loc[:, 'sentiment_score'] = dataset['rating'].apply(sentiment)

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm",disable=["ner"])

In [ ]:
valid_aspects = {
    "sound","speaker","alexa","voice","battery",
    "screen","wifi","bluetooth","setup",
    "apps","books","display","volume","music"
}

# def extract_aspects_batch(texts):

#     results = []

#     for doc in nlp.pipe(texts, batch_size=200):

#         aspects = [
#             token.lemma_.lower()
#             for token in doc
#             if token.pos_ == "NOUN" and token.lemma_.lower() in valid_aspects
#         ]

#         results.append(aspects)

#     return results

# def extract_aspects_batch(texts):
#     results = []
#     for doc in nlp.pipe(texts, batch_size=200):
#         aspects = [token.lemma_.lower() for token in doc if token.pos_ == "NOUN"]
#         results.append(aspects)
#     return results



def extract_aspects_batch(texts):

    results = []

    for doc in nlp.pipe(texts, batch_size=200):

        aspects = []

        for token in doc:
            if token.pos_ == "NOUN":
                compound = [child.text for child in token.lefts if child.dep_ == "compound"]
                compound.append(token.text)
                aspect = " ".join(compound).lower()
                aspects.append(aspect)

        results.append(aspects)

    return results

In [ ]:
alexa_df.loc[:, 'review'] = alexa_df['review'].fillna('')
echo_df.loc[:, 'review'] = echo_df['review'].fillna('')
tablet_df.loc[:, 'review'] = tablet_df['review'].fillna('')

alexa_df.loc[:, 'Aspect'] = pd.Series(extract_aspects_batch(alexa_df['review']), index=alexa_df.index)
echo_df.loc[:, 'Aspect'] = pd.Series(extract_aspects_batch(echo_df['review']), index=echo_df.index)
tablet_df.loc[:, 'Aspect'] = pd.Series(extract_aspects_batch(tablet_df['review']), index=tablet_df.index)

***ECHO***

In [ ]:
print("Echo aspects:")
print(echo_df[['Aspect']].head(20))

In [ ]:
# create working dataset
final_dataset = echo_df.copy()

# explode aspect lists
final_dataset = final_dataset.explode('Aspect')

# clean aspect text
final_dataset['Aspect'] = final_dataset['Aspect'].astype(str).str.lower().str.strip()

# remove empty aspects
final_dataset = final_dataset[final_dataset['Aspect'] != '']

In [ ]:
final_dataset['Aspect'].value_counts().head(20)

In [ ]:
echo_df['review'].str.contains('battery', case=False).sum()


In [ ]:
print(echo_df.columns)

In [ ]:
print(final_dataset['Aspect'].value_counts().head(20))

In [ ]:
final_dataset = final_dataset.explode('Aspect')

In [ ]:
valid_aspects = [
    'music','speaker','sound','voice','battery',
    'app','weather','device','light'
]

final_dataset = final_dataset[
    final_dataset['Aspect'].isin(valid_aspects)
]

In [ ]:
final_dataset['Aspect'].value_counts()

In [ ]:
final_dataset['time/date'] = pd.to_datetime(final_dataset['time/date'])
final_dataset['Month'] = final_dataset['time/date'].dt.to_period('M')

In [ ]:
aspect_month_sentiment = final_dataset.groupby(
    ['Month','Aspect']
)['sentiment_score'].mean().reset_index()

In [ ]:
print(final_dataset.columns)

In [ ]:
pivot_table = aspect_month_sentiment.pivot_table(
    index='Month',
    columns='Aspect',
    values='sentiment_score'
)

In [ ]:
# top_aspects = final_dataset['Aspect'].value_counts().head(5).index

# filtered = aspect_month_sentiment[
#     aspect_month_sentiment['Aspect'].isin(top_aspects)
# ]

# pivot_table = filtered.pivot_table(
#     index='Month',
#     columns='Aspect',
#     values='sentiment_score'
# )

In [ ]:
pivot_table.plot(figsize=(12,6))

In [ ]:
final_dataset.groupby('Aspect')['sentiment_score'].value_counts()

In [ ]:
sentiment_table = final_dataset.pivot_table(
    index='Aspect',
    columns='sentiment_score',
    aggfunc='size',
    fill_value=0
)

sentiment_table

In [ ]:
import matplotlib.pyplot as plt

sentiment_table.plot(kind='bar', figsize=(10,6))

plt.title("Aspect Sentiment Distribution (Echo)")
plt.xlabel("Aspect")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
print("Alexa aspects:")
print(alexa_df[['Aspect']].head(20))

In [ ]:
final_dataset = alexa_df.copy()

alexa_df = alexa_df.copy()

alexa_df["cleanText"] = alexa_df["review"].apply(clean_text)
alexa_df["Aspect"] = extract_aspects_batch(alexa_df["cleanText"])

alexa_df = alexa_df.explode("Aspect")


In [ ]:
aspect_mapping = {
    # battery
    "battery life": "battery",
    "charge": "battery",
    "power": "battery",

    # price
    "value": "price",
    "cost": "price",
    "deal": "price",

    # display
    "screen": "display",

    # content
    "books": "content",
    "library": "content",
    "reading": "content",

    # performance
    "speed": "performance",
    "lag": "performance",
    "experience": "performance",
    "features": "performance",

    # device
    "kindle": "device",
    "tablet": "device",
    "amazon": "device",
    "ipad": "device",
    "ereader": "device",
    "product": "device"
}

In [ ]:
# STEP 1: Apply mapping (FIX warning too)
alexa_df.loc[:, "Aspect"] = alexa_df["Aspect"].replace(aspect_mapping)


# STEP 2: Remove noise (ADD THIS - you didn’t have it properly)
noise = ["thing", "one", "something", "anything", "wife", "face"]
alexa_df = alexa_df[~alexa_df["Aspect"].isin(noise)]


# STEP 3: CHECK BEFORE FILTERING (VERY IMPORTANT)
print("After mapping:")
print(alexa_df["Aspect"].value_counts().head(20))


# STEP 4: Now align with Echo aspects
echo_aspects = ['battery', 'price', 'display', 'content', 'performance', 'device']

alexa_df = alexa_df[alexa_df["Aspect"].isin(echo_aspects)]


# STEP 5: FINAL CHECK
print("Final aspects:")
print(alexa_df["Aspect"].unique())

In [ ]:
final_dataset = final_dataset.explode('Aspect')

In [ ]:
stop_aspects = ["thing", "one", "something", "anything", "wife", "face"]

alexa_df = alexa_df[~alexa_df["Aspect"].isin(stop_aspects)]

In [ ]:
aspect_mapping = {
    # battery
    "battery life": "battery",
    "charge": "battery",
    "power": "battery",

    # price
    "value": "price",
    "cost": "price",
    "deal": "price",

    # display
    "screen": "display",

    # content
    "books": "content",
    "library": "content",
    "reading": "content",

    # performance
    "speed": "performance",
    "lag": "performance",
    "experience": "performance",
    "features": "performance",

    # device
    "kindle": "device",
    "tablet": "device",
    "amazon": "device",
    "ipad": "device",
    "ereader": "device",
    "product": "device"
}

In [ ]:
alexa_df.loc[:, "Aspect"] = alexa_df["Aspect"].replace(aspect_mapping)


In [ ]:
echo_aspects = ['battery', 'price', 'display', 'content', 'performance', 'device']

alexa_df = alexa_df[alexa_df["Aspect"].isin(echo_aspects)]

In [ ]:
print(alexa_df["Aspect"].unique())

In [ ]:
print(alexa_df["Aspect"].value_counts().head(15))

In [ ]:
final_dataset.head()

In [ ]:
final_dataset['time/date'] = pd.to_datetime(final_dataset['time/date'], format='ISO8601')

final_dataset['Month'] = final_dataset['time/date'].dt.to_period('M')

In [ ]:
pivot_table.head()

In [ ]:
import matplotlib.pyplot as plt

pivot_table.plot(figsize=(12,6))

plt.title("Aspect Sentiment Drift Over Time")
plt.xlabel("Month")
plt.ylabel("Average Sentiment Score")
plt.legend(title="Aspect")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

pivot_table.plot(figsize=(12,6))

plt.title("Aspect Sentiment Drift Over Time")
plt.xlabel("Month")
plt.ylabel("Average Sentiment Score")
plt.legend(title="Aspect")
plt.show()

In [ ]:
top_aspects = df["aspect"].value_counts().head(5).index
filtered = pivot_table[top_aspects]

filtered.plot(figsize=(12,6))

In [ ]:
pivot_table = pivot_table.sort_index()

In [ ]:
filtered.plot(figsize=(12,6), marker='o')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

sns.heatmap(
    pivot_table.T,
    cmap="RdYlGn",
    annot=False,
    linewidths=0.5
)

plt.title("Aspect Sentiment Drift Heatmap")
plt.xlabel("Month")
plt.ylabel("Aspect")

plt.show()

In [ ]:
rolling_mean = pivot_table.rolling(window=3).mean()

drift_ma = pivot_table - rolling_mean

drift_ma.head()

In [ ]:
drift_flag = drift_ma.abs() > 0.5
drift_flag.sum()

In [ ]:
drift_score = drift_ma.abs().mean()

drift_score.sort_values(ascending=False)

In [ ]:
drift_score.sort_values().plot(kind="barh", figsize=(8,5))

plt.title("Aspect Sentiment Drift Score")
plt.xlabel("Average Drift Magnitude")
plt.ylabel("Aspect")

plt.show()

**Phase 5**

In [ ]:
df["time/date"] = pd.to_datetime(df["time/date"], errors="coerce")
df["month"] = df["time/date"].dt.to_period("M")
df["time_index"] = df["month"].astype("category").cat.codes

***More Features***

In [ ]:
df_sample = df.sample(8000, random_state=42)
df_sample["time/date"] = pd.to_datetime(df_sample["time/date"], format="%d-%m-%Y", errors='coerce')
df_sample["month_numeric"] = df_sample["time/date"].dt.month
df_sample["year_numeric"] = df_sample["time/date"].dt.year

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")

In [ ]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def get_embeddings_batch(texts, batch_size=128):

    all_embeddings = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

        all_embeddings.extend(batch_embeddings)

    return all_embeddings

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
df_sample = df.sample(8000, random_state=42)

In [ ]:
import re
import spacy

# Ensure clean_text function is defined
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\\S+", "", text)
    text = re.sub(r"[^a-zA-Z\\s]", "", text)
    return text

# Load spacy model and define extract_aspects function
nlp = spacy.load("en_core_web_sm")

def extract_aspects(text):
    doc = nlp(text)
    aspects = []
    for chunk in doc.noun_chunks:
        aspect = chunk.text.lower().strip()
        if len(aspect) > 3:
            aspects.append(aspect)
    return aspects

# Re-create necessary columns for df_sample, as the df it was sampled from
# was re-initialized in cell 3GAsErp4txmS and no longer contained these.

# 1. Add sentiment_score to df_sample
def sentiment(r):
    if r >= 4:
        return 1
    elif r == 3:
        return 0
    else:
        return -1
df_sample['sentiment_score'] = df_sample['rating'].apply(sentiment)

# 2. Create reviewText, clean_review, and aspect columns
df_sample["reviewText"] = df_sample["review"].fillna("")
df_sample['clean_review'] = df_sample['reviewText'].apply(clean_text)
df_sample['aspect'] = df_sample['clean_review'].apply(extract_aspects)

# 3. Filter out rows where no aspects were found, then explode the 'aspect' column
df_sample = df_sample[df_sample['aspect'].map(len) > 0]
df_sample = df_sample.explode('aspect')

# 4. Create the aspect_text column required for embeddings
df_sample["aspect_text"] = df_sample["aspect"] + " : " + df_sample["reviewText"]

# Original code to generate embeddings
texts = df_sample["aspect_text"].tolist()
embeddings = get_embeddings_batch(texts, batch_size=128)

In [ ]:
import numpy as np

X_text = np.vstack(embeddings)

X_time = df_sample["time_index"].values.reshape(-1,1)

X = np.hstack([X_text, X_time])

In [ ]:
y = df_sample["sentiment_score"].values

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_temporal = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)
model_temporal.fit(X_train, y_train)

In [ ]:
train_acc = model_temporal.score(X_train, y_train)
test_acc = model_temporal.score(X_test, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

In [ ]:
from sklearn.metrics import classification_report

y_pred = model_temporal.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

accuracy = model_temporal.score(X_test, y_test)

plt.bar(["Temporal Model"], [accuracy])
plt.ylabel("Accuracy")
plt.title("Temporal Sentiment Model Performance")
plt.ylim(0,1)

plt.show()



**PHASE 6**



In [ ]:
drift_score = drift_ma.abs().mean()

In [ ]:
drift_df = drift_score.reset_index()
drift_df.columns = ["aspect", "drift_score"]

drift_df.sort_values(by="drift_score", ascending=False)

In [ ]:
threshold = drift_df['drift_score'].mean() # Define a threshold
drift_df["status"] = drift_df["drift_score"].apply(
    lambda x: "⚠ Emerging Negative Aspect" if x > threshold else "Stable"
)

drift_df

In [ ]:
import matplotlib.pyplot as plt

colors = ["red" if s != "Stable" else "green" for s in drift_df["status"]]

plt.figure(figsize=(10,5))
plt.bar(drift_df["aspect"], drift_df["drift_score"], color=colors)

plt.axhline(threshold, color="black", linestyle="--", label="Drift Threshold")

plt.title("Emerging Negative Aspect Detection")
plt.xlabel("Aspect")
plt.ylabel("Drift Score")

plt.legend()
plt.show()

**PHASE 7**

In [ ]:
X_baseline = X_text

In [ ]:
from sklearn.model_selection import train_test_split

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

baseline_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced"
)

baseline_model.fit(Xb_train, yb_train)

In [ ]:
baseline_accuracy = baseline_model.score(Xb_test, yb_test)

print("Baseline Accuracy:", baseline_accuracy)

In [ ]:
import matplotlib.pyplot as plt

models = ["Baseline Model", "Temporal Model"]
scores = [baseline_accuracy, accuracy]

plt.figure(figsize=(6,4))
plt.bar(models, scores)

plt.ylabel("Accuracy")
plt.title("Baseline vs Temporal Model")
plt.ylim(0,1)

plt.show()

**PHASE 8**

In [ ]:
import matplotlib.pyplot as plt

pivot_table.plot(figsize=(12,6), marker='o')

plt.title("Aspect Sentiment Trend Over Time")
plt.xlabel("Month")
plt.ylabel("Average Sentiment Score")
plt.legend(title="Aspect")

plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(12,6))

sns.heatmap(
    pivot_table.T,
    cmap="RdYlGn",
    linewidths=0.5
)

plt.title("Aspect Sentiment Heatmap Over Time")
plt.xlabel("Month")
plt.ylabel("Aspect")

plt.show()

In [ ]:
drift_score.sort_values().plot(
    kind="barh",
    figsize=(8,5)
)

plt.title("Aspect Sentiment Drift Score")
plt.xlabel("Drift Magnitude")
plt.ylabel("Aspect")

plt.show()

In [ ]:
colors = ["red" if s != "Stable" else "green" for s in drift_df["status"]]

plt.figure(figsize=(10,5))

plt.bar(drift_df["aspect"], drift_df["drift_score"], color=colors)

plt.axhline(threshold, linestyle="--", color="black", label="Drift Threshold")

plt.title("Emerging Negative Aspect Detection")
plt.xlabel("Aspect")
plt.ylabel("Drift Score")

plt.legend()

plt.show()

In [ ]:
models = ["Baseline Model", "Temporal Model"]
scores = [baseline_accuracy, accuracy]

plt.figure(figsize=(6,4))

plt.bar(models, scores)

plt.ylabel("Accuracy")
plt.title("Baseline vs Temporal Model Performance")
plt.ylim(0,1)

# Add accuracy numbers on top of bars
for i, v in enumerate(scores):
    plt.text(i, v + 0.01, str(round(v,2)), ha='center')

plt.show()

**DOWNLOAD GRAPH**

In [ ]:
plt.savefig("sentiment_trend.png", dpi=300)

In [ ]:
!ls

In [ ]:
from google.colab import files
files.download("sentiment_trend.png")


In [ ]:

plt.savefig("heatmap.png", dpi=300)


In [ ]:
from google.colab import files
files.download("heatmap.png")


In [ ]:
plt.savefig("drift_score.png", dpi=300)

In [ ]:
from google.colab import files
files.download("drift_score.png")

In [ ]:
plt.savefig("emerging_aspects.png", dpi=300)

In [ ]:
from google.colab import files
files.download("emerging_aspects.png")

In [ ]:
plt.savefig("model_comparison.png", dpi=300)

In [ ]:
from google.colab import files
files.download("model_comparison.png")

**SAMPLE FILE**

In [ ]:
df_sample = df.head(100)


In [ ]:
df_sample.to_csv("sample_reviews.csv", index=False)


In [ ]:
from google.colab import files
files.download("sample_reviews.csv")

***DASHBOARD***

In [ ]:
!pip install streamlit

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

In [ ]:
!streamlit run app.py

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.title("Temporal Aspect Sentiment Drift Dashboard")

st.write("Analysis of sentiment changes across product aspects over time.")

# Load data
pivot_table = pd.read_csv("pivot_aspect_sentiment_monthly.csv", index_col=0)
drift_df = pd.read_csv("drift_results.csv")

# Sentiment Trend
st.subheader("Aspect Sentiment Trend Over Time")

fig, ax = plt.subplots(figsize=(10,5))
pivot_table.plot(ax=ax)
st.pyplot(fig)

# Drift Score
st.subheader("Aspect Sentiment Drift Score")

fig2, ax2 = plt.subplots()
ax2.barh(drift_df["aspect"], drift_df["drift_score"])
ax2.set_xlabel("Drift Score")
st.pyplot(fig2)

# Emerging Aspects
st.subheader("Emerging Negative Aspects")

emerging = drift_df[drift_df["status"] != "Stable"]
st.dataframe(emerging)

In [ ]:
pivot_table.to_csv("pivot_aspect_sentiment_monthly.csv")

drift_df.to_csv("drift_results.csv")

In [ ]:
!streamlit run app.py

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
!pip install streamlit pyngrok

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.title("Temporal Aspect Sentiment Drift Dashboard")

pivot_table = pd.read_csv("pivot_aspect_sentiment_monthly.csv", index_col=0)
drift_df = pd.read_csv("drift_results.csv")

st.subheader("Aspect Sentiment Trend")
fig, ax = plt.subplots(figsize=(10,5))
pivot_table.plot(ax=ax)
st.pyplot(fig)

st.subheader("Aspect Drift Score")
fig2, ax2 = plt.subplots()
ax2.barh(drift_df["aspect"], drift_df["drift_score"])
st.pyplot(fig2)

st.subheader("Emerging Negative Aspects")
emerging = drift_df[drift_df["status"] != "Stable"]
st.dataframe(emerging)

In [ ]:
from pyngrok import ngrok
import subprocess

public_url = ngrok.connect(8501)
print("Open this URL:", public_url)

!streamlit run app.py &